# 03 · Análisis y Reportes — Caso Familia Miranda
**Proyecto:** IW Resource Management – Caso Familia Miranda  
**Autor:** Diego Ballesteros  
**Fecha:** 2025  
**Notebook:** 3 de 4

---

## Objetivo
Responder todas las preguntas del caso de negocio mediante consultas SQL sobre el modelo relacional construido en el notebook 02. Cada sección corresponde directamente a una pregunta del enunciado.

**Meses analizados:** Agosto 2023 · Septiembre 2023

---

## Índice de reportes
1. Setup y conexión
2. Ejecución presupuestal — planeado vs. real por rubro
3. ¿Cuánto gana la familia al mes?
4. ¿Cuánto gasta la familia al mes?
5. ¿Está quedando suficiente dinero para ahorrar?
6. Top 3 rubros con mayor sobreejercicio
7. Medio de pago preferido por miembro
8. Gastos NO registrados en rubros presupuestados
9. Rubros presupuestados NO utilizados en el mes
10. Resumen ejecutivo

---
## 1. Setup y conexión

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from sqlalchemy import create_engine, text
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# ── Estilo visual consistente ─────────────────────────────────────────────────
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi']     = 120
plt.rcParams['font.family']    = 'DejaVu Sans'
plt.rcParams['axes.titlesize'] = 13
COLORES = {'papa': '#4C72B0', 'mama': '#DD8452', 'hijo': '#55A868'}

# ── Conexión ──────────────────────────────────────────────────────────────────
BASE_DIR = Path('..')
DB_PATH  = BASE_DIR / 'data' / 'familia_miranda.db'
engine   = create_engine(f'sqlite:///{DB_PATH}', echo=False)

def run_query(sql: str) -> pd.DataFrame:
    """Ejecuta una consulta SQL y retorna un DataFrame."""
    with engine.connect() as conn:
        return pd.read_sql(text(sql), conn)

def fmt_cop(val) -> str:
    """Formatea un número como pesos colombianos."""
    return f'$ {val:>15,.0f}'

print('✅ Conexión establecida')
print(f'   Base de datos: {DB_PATH.resolve()}')

---
## 2. Ejecución presupuestal — Planeado vs. Real por rubro

> **Pregunta:** Obtenga la información de los gastos de cada miembro, haga la sumatoria en el campo Real cruzando por el nombre del rubro. ¿Cuál es el resultado de la ejecución presupuestal?

Esta consulta es el corazón del análisis: une el presupuesto con los gastos reales de los tres miembros para calcular la diferencia por rubro.

In [ ]:
MES = '2023-09'  # Mes de análisis principal (cambiar a '2023-08' para agosto)

sql_ejecucion = f"""
    SELECT
        r.nombre_rubro                          AS rubro,
        p.valor_planeado                        AS planeado,
        COALESCE(SUM(g_papa.valor),  0)         AS gasto_papa,
        COALESCE(SUM(g_mama.valor),  0)         AS gasto_mama,
        COALESCE(SUM(g_hijo.valor),  0)         AS gasto_hijo,
        COALESCE(SUM(g_papa.valor),  0)
            + COALESCE(SUM(g_mama.valor), 0)
            + COALESCE(SUM(g_hijo.valor), 0)    AS total_real,
        p.valor_planeado - (
            COALESCE(SUM(g_papa.valor), 0)
            + COALESCE(SUM(g_mama.valor), 0)
            + COALESCE(SUM(g_hijo.valor), 0)
        )                                       AS diferencia
    FROM presupuesto p
    JOIN rubros r ON p.id_rubro = r.id_rubro
    LEFT JOIN gastos g_papa
        ON g_papa.id_rubro = p.id_rubro
        AND g_papa.mes = p.mes
        AND g_papa.id_miembro = (SELECT id_miembro FROM miembros WHERE nombre = 'papa')
    LEFT JOIN gastos g_mama
        ON g_mama.id_rubro = p.id_rubro
        AND g_mama.mes = p.mes
        AND g_mama.id_miembro = (SELECT id_miembro FROM miembros WHERE nombre = 'mama')
    LEFT JOIN gastos g_hijo
        ON g_hijo.id_rubro = p.id_rubro
        AND g_hijo.mes = p.mes
        AND g_hijo.id_miembro = (SELECT id_miembro FROM miembros WHERE nombre = 'hijo')
    WHERE p.mes = '{MES}'
    GROUP BY r.nombre_rubro, p.valor_planeado
    ORDER BY diferencia ASC
"""

df_ejec = run_query(sql_ejecucion)

# Formatear para display
df_display = df_ejec.copy()
for col in ['planeado','gasto_papa','gasto_mama','gasto_hijo','total_real','diferencia']:
    df_display[col] = df_display[col].apply(lambda x: f'$ {x:>12,.0f}')

print(f'EJECUCIÓN PRESUPUESTAL — {MES}')
print('=' * 95)
print(df_display.to_string(index=False))

In [ ]:
# ── Visualización: planeado vs real por rubro (top 15 por gasto real) ─────────
df_plot = df_ejec[df_ejec['total_real'] > 0].nlargest(15, 'total_real')

fig, ax = plt.subplots(figsize=(12, 6))
x = range(len(df_plot))
width = 0.38

bars1 = ax.bar([i - width/2 for i in x], df_plot['planeado'],  width, label='Planeado',  color='#4C72B0', alpha=0.85)
bars2 = ax.bar([i + width/2 for i in x], df_plot['total_real'], width, label='Real',     color='#DD8452', alpha=0.85)

ax.set_xticks(list(x))
ax.set_xticklabels(df_plot['rubro'], rotation=35, ha='right', fontsize=9)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'${v/1e6:.1f}M'))
ax.set_title(f'Planeado vs. Real por Rubro — {MES}', fontsize=14, pad=12)
ax.set_ylabel('Valor (COP)')
ax.legend()
plt.tight_layout()
plt.savefig(BASE_DIR / 'data' / 'processed' / 'ejecucion_presupuestal.png', bbox_inches='tight')
plt.show()
print('📊 Gráfico guardado en data/processed/')

---
## 3. ¿Cuánto dinero gana la familia al mes?

> **Pregunta:** ¿Cuánto dinero gana la familia al mes?

In [ ]:
sql_ingresos = f"""
    SELECT
        m.nombre                AS miembro,
        SUM(g.valor)            AS ingreso_real,
        MAX(p.valor_planeado)   AS ingreso_planeado
    FROM gastos g
    JOIN miembros m ON g.id_miembro = m.id_miembro
    JOIN rubros   r ON g.id_rubro   = r.id_rubro
    JOIN presupuesto p
        ON p.id_rubro = g.id_rubro AND p.mes = g.mes
    WHERE g.mes = '{MES}'
      AND LOWER(r.nombre_rubro) LIKE '%contrato%'
    GROUP BY m.nombre
    ORDER BY ingreso_real DESC
"""

df_ingresos = run_query(sql_ingresos)
total_ingreso = df_ingresos['ingreso_real'].sum()

print(f'INGRESOS FAMILIA MIRANDA — {MES}')
print('─' * 45)
for _, row in df_ingresos.iterrows():
    print(f"  {row['miembro']:<8}  {fmt_cop(row['ingreso_real'])}")
print('─' * 45)
print(f"  {'TOTAL':<8}  {fmt_cop(total_ingreso)}")
print()
print(f'📌 La familia Miranda gana {fmt_cop(total_ingreso)} en {MES}.')

---
## 4. ¿Cuánto gasta la familia al mes?

> **Pregunta:** ¿Cuánto dinero gastó la familia al mes?

In [ ]:
sql_gastos_total = f"""
    SELECT
        m.nombre        AS miembro,
        COUNT(*)        AS num_transacciones,
        SUM(g.valor)    AS total_gasto
    FROM gastos g
    JOIN miembros m ON g.id_miembro = m.id_miembro
    WHERE g.mes = '{MES}'
      AND LOWER(COALESCE(g.categoria_origen, '')) NOT LIKE '%contrato%'
    GROUP BY m.nombre
    ORDER BY total_gasto DESC
"""

df_gastos_total = run_query(sql_gastos_total)
total_gasto = df_gastos_total['total_gasto'].sum()

print(f'GASTOS FAMILIA MIRANDA — {MES}')
print('─' * 55)
for _, row in df_gastos_total.iterrows():
    print(f"  {row['miembro']:<8}  {row['num_transacciones']:>3} transacciones  {fmt_cop(row['total_gasto'])}")
print('─' * 55)
print(f"  {'TOTAL':<8}  {df_gastos_total['num_transacciones'].sum():>3} transacciones  {fmt_cop(total_gasto)}")
print()
print(f'📌 La familia gastó {fmt_cop(total_gasto)} en {MES}.')

---
## 5. ¿Está quedando suficiente dinero para ahorrar?

> **Pregunta:** ¿La familia está gastando más de lo que gana? ¿Está quedando suficiente dinero para ahorrar?

In [ ]:
# Comparar ingreso vs gasto para ambos meses
sql_flujo = """
    SELECT
        g.mes,
        SUM(CASE WHEN LOWER(r.nombre_rubro) LIKE '%contrato%' THEN g.valor ELSE 0 END) AS ingresos,
        SUM(CASE WHEN LOWER(r.nombre_rubro) NOT LIKE '%contrato%' THEN g.valor ELSE 0 END) AS gastos
    FROM gastos g
    LEFT JOIN rubros r ON g.id_rubro = r.id_rubro
    GROUP BY g.mes
    ORDER BY g.mes
"""

df_flujo = run_query(sql_flujo)
df_flujo['ahorro']        = df_flujo['ingresos'] - df_flujo['gastos']
df_flujo['tasa_ahorro_%'] = (df_flujo['ahorro'] / df_flujo['ingresos'] * 100).round(1)

print('FLUJO DE CAJA — INGRESOS vs. GASTOS')
print('═' * 70)
for _, row in df_flujo.iterrows():
    signo = '🟢' if row['ahorro'] >= 0 else '🔴'
    print(f"  {row['mes']}")
    print(f"    Ingresos  : {fmt_cop(row['ingresos'])}")
    print(f"    Gastos    : {fmt_cop(row['gastos'])}")
    print(f"    Ahorro    : {fmt_cop(row['ahorro'])}  {signo}  ({row['tasa_ahorro_%']}% del ingreso)")
    print()

# Interpretación
for _, row in df_flujo.iterrows():
    if row['ahorro'] < 0:
        print(f"⚠️  {row['mes']}: La familia gasta MÁS de lo que gana. Déficit de {fmt_cop(abs(row['ahorro']))}")
    elif row['tasa_ahorro_%'] < 10:
        print(f"⚠️  {row['mes']}: Ahorro insuficiente ({row['tasa_ahorro_%']}%). Se recomienda mínimo 20%.")
    else:
        print(f"✅  {row['mes']}: La familia tiene capacidad de ahorro ({row['tasa_ahorro_%']}%).")

In [ ]:
# ── Visualización: flujo de caja comparativo ──────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 5))
meses   = df_flujo['mes'].tolist()
x       = range(len(meses))
width   = 0.28

ax.bar([i - width for i in x], df_flujo['ingresos'], width, label='Ingresos', color='#55A868', alpha=0.9)
ax.bar([i          for i in x], df_flujo['gastos'],   width, label='Gastos',   color='#DD8452', alpha=0.9)
ax.bar([i + width  for i in x], df_flujo['ahorro'],   width, label='Ahorro',
       color=['#4C72B0' if v >= 0 else '#C44E52' for v in df_flujo['ahorro']], alpha=0.9)

ax.set_xticks(list(x))
ax.set_xticklabels(meses)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'${v/1e6:.1f}M'))
ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
ax.set_title('Flujo de Caja Familiar — Ingresos vs. Gastos vs. Ahorro', pad=12)
ax.set_ylabel('Valor (COP)')
ax.legend()
plt.tight_layout()
plt.savefig(BASE_DIR / 'data' / 'processed' / 'flujo_caja.png', bbox_inches='tight')
plt.show()

---
## 6. Top 3 rubros con mayor sobreejercicio presupuestal

> **Pregunta:** ¿Cuál es el top 3 de los rubros que están gastando más dinero del presupuestado?

In [ ]:
sql_top_sobreejecucion = f"""
    SELECT
        r.nombre_rubro          AS rubro,
        p.valor_planeado        AS planeado,
        SUM(g.valor)            AS total_real,
        SUM(g.valor) - p.valor_planeado          AS exceso,
        ROUND((SUM(g.valor) - p.valor_planeado)
            / p.valor_planeado * 100, 1)         AS exceso_pct
    FROM presupuesto p
    JOIN rubros r ON p.id_rubro = r.id_rubro
    JOIN gastos g ON g.id_rubro = p.id_rubro AND g.mes = p.mes
    WHERE p.mes = '{MES}'
      AND LOWER(r.nombre_rubro) NOT LIKE '%contrato%'
    GROUP BY r.nombre_rubro, p.valor_planeado
    HAVING SUM(g.valor) > p.valor_planeado
    ORDER BY exceso DESC
    LIMIT 3
"""

df_top3 = run_query(sql_top_sobreejecucion)

print(f'TOP 3 RUBROS CON MAYOR SOBREEJERCICIO — {MES}')
print('═' * 65)
for i, row in df_top3.iterrows():
    print(f"  #{i+1} {row['rubro']}")
    print(f"      Planeado  : {fmt_cop(row['planeado'])}")
    print(f"      Real      : {fmt_cop(row['total_real'])}")
    print(f"      Exceso    : {fmt_cop(row['exceso'])}  (+{row['exceso_pct']}%)")
    print()

In [ ]:
# ── Visualización: Top 3 sobreejercicio ───────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 4))
colores_bar = ['#C44E52', '#DD8452', '#CCB974']

bars = ax.barh(df_top3['rubro'], df_top3['exceso'], color=colores_bar, alpha=0.9)
for bar, pct in zip(bars, df_top3['exceso_pct']):
    ax.text(bar.get_width() * 1.01, bar.get_y() + bar.get_height()/2,
            f'+{pct}%', va='center', fontsize=10, color='#333')

ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'${v/1e6:.1f}M'))
ax.set_title(f'Top 3 Rubros — Mayor Sobreejercicio ({MES})', pad=12)
ax.set_xlabel('Exceso sobre lo presupuestado (COP)')
ax.invert_yaxis()
plt.tight_layout()
plt.savefig(BASE_DIR / 'data' / 'processed' / 'top3_sobreejercicio.png', bbox_inches='tight')
plt.show()

---
## 7. Medio de pago preferido por miembro

> **Pregunta:** ¿Cuál es el tipo de pago preferido de cada miembro de la familia?

In [ ]:
# Medio de pago preferido = el que más veces se usó (por frecuencia, no por monto)
sql_pago_preferido = f"""
    SELECT
        m.nombre        AS miembro,
        g.forma_pago,
        COUNT(*)        AS frecuencia,
        SUM(g.valor)    AS monto_total
    FROM gastos g
    JOIN miembros m ON g.id_miembro = m.id_miembro
    WHERE g.mes = '{MES}'
      AND LOWER(COALESCE(g.categoria_origen,'')) NOT LIKE '%contrato%'
    GROUP BY m.nombre, g.forma_pago
    ORDER BY m.nombre, frecuencia DESC
"""

df_pago = run_query(sql_pago_preferido)

print(f'MEDIO DE PAGO POR MIEMBRO — {MES}')
print('═' * 60)
for miembro in df_pago['miembro'].unique():
    df_m = df_pago[df_pago['miembro'] == miembro].reset_index(drop=True)
    preferido = df_m.iloc[0]
    print(f"  {miembro.upper()}")
    print(f"    ★ Preferido: {preferido['forma_pago']:<15} ({preferido['frecuencia']} transacciones)")
    for _, row in df_m.iterrows():
        pct = row['frecuencia'] / df_m['frecuencia'].sum() * 100
        barra = '█' * int(pct / 5)
        print(f"      {row['forma_pago']:<15} {barra:<20} {pct:>5.1f}%  {fmt_cop(row['monto_total'])}")
    print()

In [ ]:
# ── Visualización: distribución de medios de pago por miembro ─────────────────
fig, axes = plt.subplots(1, 3, figsize=(12, 4))

for ax, miembro in zip(axes, ['papa', 'mama', 'hijo']):
    df_m = df_pago[df_pago['miembro'] == miembro]
    if df_m.empty:
        ax.set_visible(False)
        continue
    ax.pie(
        df_m['frecuencia'],
        labels=df_m['forma_pago'],
        autopct='%1.1f%%',
        startangle=90,
        colors=sns.color_palette('muted', len(df_m))
    )
    ax.set_title(f'{miembro.capitalize()}', fontsize=12)

fig.suptitle(f'Distribución de Medios de Pago por Miembro — {MES}', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig(BASE_DIR / 'data' / 'processed' / 'medios_pago.png', bbox_inches='tight')
plt.show()

---
## 8. Gastos NO registrados en rubros presupuestados

> **Pregunta:** ¿Cuáles gastos NO están en rubros presupuestados?

In [ ]:
sql_sin_rubro = f"""
    SELECT
        m.nombre            AS miembro,
        g.fecha,
        g.descripcion,
        g.categoria_origen  AS categoria,
        g.valor,
        g.forma_pago
    FROM gastos g
    JOIN miembros m ON g.id_miembro = m.id_miembro
    WHERE g.mes = '{MES}'
      AND g.id_rubro IS NULL
    ORDER BY g.valor DESC
"""

df_sin_rubro = run_query(sql_sin_rubro)
total_sin_rubro = df_sin_rubro['valor'].sum()

print(f'GASTOS SIN RUBRO PRESUPUESTADO — {MES}')
print(f'Total: {len(df_sin_rubro)} transacciones por {fmt_cop(total_sin_rubro)}')
print('═' * 80)

# Agrupar por categoría para resumen
resumen = (
    df_sin_rubro
    .groupby('categoria')
    .agg(transacciones=('valor','count'), total=('valor','sum'))
    .sort_values('total', ascending=False)
)
resumen['total_fmt'] = resumen['total'].apply(fmt_cop)
print(resumen[['transacciones','total_fmt']].to_string())
print()
print('📌 Estos gastos representan categorías que la familia debería considerar')
print('   agregar al presupuesto en meses futuros (especialmente PRESTAMO e inversiones).')

---
## 9. Rubros presupuestados NO utilizados en el mes

> **Pregunta:** ¿Cuáles rubros presupuestados no fueron utilizados durante el mes?

In [ ]:
sql_rubros_sin_gasto = f"""
    SELECT
        r.nombre_rubro      AS rubro,
        p.valor_planeado    AS presupuestado
    FROM presupuesto p
    JOIN rubros r ON p.id_rubro = r.id_rubro
    LEFT JOIN gastos g
        ON g.id_rubro = p.id_rubro
        AND g.mes     = p.mes
    WHERE p.mes = '{MES}'
      AND g.id_gasto IS NULL
      AND LOWER(r.nombre_rubro) NOT LIKE '%contrato%'
    ORDER BY p.valor_planeado DESC
"""

df_sin_gasto = run_query(sql_rubros_sin_gasto)
total_no_usado = df_sin_gasto['presupuestado'].sum()

print(f'RUBROS PRESUPUESTADOS SIN GASTO REGISTRADO — {MES}')
print(f'({len(df_sin_gasto)} rubros · {fmt_cop(total_no_usado)} no ejecutados)')
print('═' * 55)
for _, row in df_sin_gasto.iterrows():
    print(f"  {row['rubro']:<40} {fmt_cop(row['presupuestado'])}")
print()
print('📌 Dinero presupuestado no ejecutado puede reasignarse a rubros con déficit,')
print('   o acumularse como ahorro adicional si el gasto no es necesario.')

---
## 10. Resumen ejecutivo

Consolidación de los hallazgos principales para presentación ante el evaluador.

In [ ]:
print('=' * 65)
print(f'  RESUMEN EJECUTIVO — FAMILIA MIRANDA — {MES}')
print('=' * 65)

# Ingresos
total_ing = run_query(f"""
    SELECT SUM(g.valor) FROM gastos g
    JOIN rubros r ON g.id_rubro = r.id_rubro
    WHERE g.mes='{MES}' AND LOWER(r.nombre_rubro) LIKE '%contrato%'
""").iloc[0,0] or 0

# Gastos
total_gto = run_query(f"""
    SELECT SUM(g.valor) FROM gastos g
    WHERE g.mes='{MES}'
      AND LOWER(COALESCE(g.categoria_origen,'')) NOT LIKE '%contrato%'
""").iloc[0,0] or 0

ahorro    = total_ing - total_gto
tasa_ah   = ahorro / total_ing * 100 if total_ing > 0 else 0

print(f"  Ingresos totales        : {fmt_cop(total_ing)}")
print(f"  Gastos totales          : {fmt_cop(total_gto)}")
print(f"  Ahorro neto             : {fmt_cop(ahorro)}  ({tasa_ah:.1f}% del ingreso)")
print(f"  Estado                  : {'✅ Superávit' if ahorro >= 0 else '🔴 Déficit'}")
print()
print(f"  Gastos sin presupuesto  : {len(df_sin_rubro)} transacciones · {fmt_cop(total_sin_rubro)}")
print(f"  Rubros sin usar         : {len(df_sin_gasto)} rubros · {fmt_cop(total_no_usado)} disponibles")
print()
print('  Top 3 rubros sobre ejecutados:')
for i, row in df_top3.iterrows():
    print(f"    #{i+1} {row['rubro']:<35} +{row['exceso_pct']}%")
print()
print('=' * 65)
print('✅ Notebook 03 completado — continuar con 04_wordcloud_bonus.ipynb')